# 05 · Annotation associations, density and legacy plots

**ScGeo 0.2.0 learning track.** Explore count-conserving flow diagrams, common-grid density and the upgraded mapping, distribution and consensus views. All labels and coordinates here are synthetic.

Use the [separate tutorial environment](README.md). Run cells in order in a fresh kernel. Outputs are written to a new ignored results directory; manuscript inputs and outputs are not used.

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import scgeo as sg
from IPython.display import display

cwd = Path.cwd().resolve()
root = next((p for p in (cwd, *cwd.parents)
             if (p / "scripts/scgeo_v020_examples/SOURCE_PART1.json").is_file()), None)
if root is None:
    raise RuntimeError("Open this notebook from the scgeo-notebooks checkout.")
examples = root / "scripts/scgeo_v020_examples"
if str(examples) not in sys.path:
    sys.path.insert(0, str(examples))  # only tutorial helpers, not package source
from tutorial_support import setup, save_preview, show_gallery, FORMATS
out = setup(root, "05_associations_density_legacy")

In [ ]:
from plotting_gallery import make_gallery_data
from legacy_plotting_gallery import run as legacy_gallery
adata = make_gallery_data(seed=17)
adata.obs["mapped_state"] = adata.obs["state"].astype(str)
changed = adata.obs.index[::11]
adata.obs.loc[changed, "mapped_state"] = "Alternative label"
columns = ["state", "mapped_state"]
display(pd.crosstab(adata.obs["state"], adata.obs["mapped_state"]))
for name, fn in [("alluvial", sg.pl.state_flow_alluvial), ("chord", sg.pl.state_flow_chord)]:
    result = fn(adata, columns=columns, min_count=2, color_by="source",
                return_result=True, show=False)
    save_preview(result, out, name)

## Interactive Sankey and density

Links are the same-cell cross-tabulation of supplied labels, not cell lineage or elapsed time. Filtering hides links, not observations. Plotly exports its interactive Sankey natively; Matplotlib uses PlotResult.

In [ ]:
fig, nodes, links = sg.pl.state_flow_sankey(adata, columns=columns,
    min_count=2, return_data=True, show=False)
nodes.to_csv(out / "sankey_nodes.csv", index=False)
links.to_csv(out / "sankey_links.csv", index=False)
fig.write_html(out / "sankey.html", include_plotlyjs=True)
fig.write_json(out / "sankey.json")
display(links.head())
result = sg.pl.embedding_density(adata, "state", basis="display",
    gridsize=60, contour=True, smooth_k=3, log1p=True, normalize="global",
    return_result=True, show=False)
save_preview(result, out, "common_density")

## Revisit legacy visualizations

The pinned legacy gallery exercises density overlap, delta ranking, mapping QC, consensus structure and the display-space compass, on the same synthetic fixture. A mapping score constructed for this example is not an accuracy estimate.

In [ ]:
legacy_gallery(out / "legacy_gallery", formats=FORMATS)
show_gallery(out / "legacy_gallery", limit=2)

See [state associations and density](https://github.com/liuifrec/scgeo/blob/v0.2.0/docs/STATE_FLOW_AND_DENSITY.md).